# View Robustness Analysis

Box plots showing precision@k and recall@k variance across views.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
from matplotlib.axes import Axes
from plotting_defaults import COLOR_MAP, delete_tex_cache, save_and_measure, set_style

from fedaugment.utils import cd_to_git_root

cd_to_git_root()
delete_tex_cache()
set_style()

plots_dir = Path("analysis/plots/view-robustness")
plots_dir.mkdir(exist_ok=True)

2026-04-22 17:26:56.792 | DEBUG    | fedaugment.utils.misc:cd_to_git_root:154 - Changed working directory to: /home/lbhm/Projects/fedaugment


In [2]:
# Load results per dataset and experiment group
base_path = Path("logs/view-robustness")

dataset_dfs: dict[str, pl.DataFrame] = {}
for dataset_path in base_path.iterdir():
    if not dataset_path.is_dir():
        continue
    dataset = dataset_path.name

    df_list: list[pl.DataFrame] = []
    for exp_group_path in dataset_path.iterdir():
        if not exp_group_path.is_dir():
            continue

        df_list.extend(
            pl.read_csv(f).with_columns(
                pl.lit(exp_group_path.name).alias("exp_group"),
                pl.lit(f.stem.split("-")[1]).alias("method"),
            )
            for f in exp_group_path.glob("*.csv")
        )

    dataset_dfs[dataset] = pl.concat(df_list).with_columns(pl.lit(dataset).alias("dataset"))

print(f"Loaded {len(dataset_dfs)} datasets")
dataset_dfs[next(iter(dataset_dfs.keys()))].head()

Loaded 5 datasets


query_id,precision@1,recall@1,max_precision@1,max_recall@1,hits@1,mrr@1,ap@1,precision@5,recall@5,max_precision@5,max_recall@5,hits@5,mrr@5,ap@5,precision@10,recall@10,max_precision@10,max_recall@10,hits@10,mrr@10,ap@10,precision@15,recall@15,max_precision@15,max_recall@15,hits@15,mrr@15,ap@15,precision@20,recall@20,max_precision@20,max_recall@20,hits@20,mrr@20,ap@20,precision@25,…,hits@30,mrr@30,ap@30,precision@35,recall@35,max_precision@35,max_recall@35,hits@35,mrr@35,ap@35,precision@40,recall@40,max_precision@40,max_recall@40,hits@40,mrr@40,ap@40,precision@45,recall@45,max_precision@45,max_recall@45,hits@45,mrr@45,ap@45,precision@50,recall@50,max_precision@50,max_recall@50,hits@50,mrr@50,ap@50,view,view_idx,ckpt_name,exp_group,method,dataset
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,str,str,str,str
"""agency_spending_noisy_0.pq::AG…",0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.4,1.0,0.4,1.0,1.0,0.5,0.583333,0.2,1.0,0.2,1.0,1.0,0.5,0.583333,0.133333,1.0,0.133333,1.0,1.0,0.5,0.583333,0.1,1.0,0.1,1.0,1.0,0.5,0.583333,0.08,…,1.0,0.5,0.583333,0.057143,1.0,0.057143,1.0,1.0,0.5,0.583333,0.05,1.0,0.05,1.0,1.0,0.5,0.583333,0.044444,1.0,0.044444,1.0,1.0,0.5,0.583333,0.04,1.0,0.04,1.0,1.0,0.5,0.583333,"""distilroberta-dj_adpt""",0,"""webtable-cl_default-v=12-disti…","""12_views""","""cl_default""","""omnimatch_city_test"""
"""agency_spending_noisy_0.pq::AG…",0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.4,1.0,0.4,1.0,1.0,0.5,0.5,0.2,1.0,0.2,1.0,1.0,0.5,0.5,0.133333,1.0,0.133333,1.0,1.0,0.5,0.5,0.1,1.0,0.1,1.0,1.0,0.5,0.5,0.08,…,1.0,0.5,0.5,0.057143,1.0,0.057143,1.0,1.0,0.5,0.5,0.05,1.0,0.05,1.0,1.0,0.5,0.5,0.044444,1.0,0.044444,1.0,1.0,0.5,0.5,0.04,1.0,0.04,1.0,1.0,0.5,0.5,"""distilroberta-dj_adpt""",0,"""webtable-cl_default-v=12-disti…","""12_views""","""cl_default""","""omnimatch_city_test"""
"""agency_spending_noisy_0.pq::BU…",0.0,0.0,1.0,0.333333,0.0,0.0,0.0,0.6,1.0,0.6,1.0,1.0,0.5,0.638889,0.3,1.0,0.3,1.0,1.0,0.5,0.638889,0.2,1.0,0.2,1.0,1.0,0.5,0.638889,0.15,1.0,0.15,1.0,1.0,0.5,0.638889,0.12,…,1.0,0.5,0.638889,0.085714,1.0,0.085714,1.0,1.0,0.5,0.638889,0.075,1.0,0.075,1.0,1.0,0.5,0.638889,0.066667,1.0,0.066667,1.0,1.0,0.5,0.638889,0.06,1.0,0.06,1.0,1.0,0.5,0.638889,"""distilroberta-dj_adpt""",0,"""webtable-cl_default-v=12-disti…","""12_views""","""cl_default""","""omnimatch_city_test"""
"""agency_spending_noisy_0.pq::PU…",0.0,0.0,1.0,0.5,0.0,0.0,0.0,0.4,1.0,0.4,1.0,1.0,0.5,0.5,0.2,1.0,0.2,1.0,1.0,0.5,0.5,0.133333,1.0,0.133333,1.0,1.0,0.5,0.5,0.1,1.0,0.1,1.0,1.0,0.5,0.5,0.08,…,1.0,0.5,0.5,0.057143,1.0,0.057143,1.0,1.0,0.5,0.5,0.05,1.0,0.05,1.0,1.0,0.5,0.5,0.044444,1.0,0.044444,1.0,1.0,0.5,0.5,0.04,1.0,0.04,1.0,1.0,0.5,0.5,"""distilroberta-dj_adpt""",0,"""webtable-cl_default-v=12-disti…","""12_views""","""cl_default""","""omnimatch_city_test"""
"""agency_spending_noisy_1.pq::BU…",1.0,0.25,1.0,0.25,1.0,1.0,0.25,0.6,0.75,0.8,1.0,1.0,1.0,0.6875,0.3,0.75,0.4,1.0,1.0,1.0,0.6875,0.266667,1.0,0.266667,1.0,1.0,1.0,0.758929,0.2,1.0,0.2,1.0,1.0,1.0,0.758929,0.16,…,1.0,1.0,0.758929,0.114286,1.0,0.114286,1.0,1.0,1.0,0.758929,0.1,1.0,0.1,1.0,1.0,1.0,0.758929,0.088889,1.0,0.088889,1.0,1.0,1.0,0.758929,0.08,1.0,0.08,1.0,1.0,1.0,0.758929,"""distilroberta-dj_adpt""",0,"""webtable-cl_default-v=12-disti…","""12_views""","""cl_default""","""omnimatch_city_test"""


In [3]:
# Compute per-view average metrics (mean across queries per view)
per_view_means_dict: dict[str, pl.DataFrame] = {}
for dataset, dataset_df in dataset_dfs.items():
    metric_cols = [c for c in dataset_df.columns if "@" in c]
    per_view_mean = dataset_df.group_by(["exp_group", "dataset", "method", "view"]).agg(
        [pl.col(c).mean().alias(c) for c in metric_cols]
    )
    per_view_means_dict[dataset] = per_view_mean

per_view_means_dict[next(iter(per_view_means_dict.keys()))].head()

exp_group,dataset,method,view,precision@1,recall@1,max_precision@1,max_recall@1,hits@1,mrr@1,ap@1,precision@5,recall@5,max_precision@5,max_recall@5,hits@5,mrr@5,ap@5,precision@10,recall@10,max_precision@10,max_recall@10,hits@10,mrr@10,ap@10,precision@15,recall@15,max_precision@15,max_recall@15,hits@15,mrr@15,ap@15,precision@20,recall@20,max_precision@20,max_recall@20,hits@20,…,mrr@25,ap@25,precision@30,recall@30,max_precision@30,max_recall@30,hits@30,mrr@30,ap@30,precision@35,recall@35,max_precision@35,max_recall@35,hits@35,mrr@35,ap@35,precision@40,recall@40,max_precision@40,max_recall@40,hits@40,mrr@40,ap@40,precision@45,recall@45,max_precision@45,max_recall@45,hits@45,mrr@45,ap@45,precision@50,recall@50,max_precision@50,max_recall@50,hits@50,mrr@50,ap@50
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""12_views""","""omnimatch_city_test""","""la2m_nopca""","""kalm_instruct_v2_5-dj_adpt""",0.545259,0.188433,1.0,0.462151,0.545259,0.545259,0.188433,0.476293,0.806345,0.581897,0.96934,0.943966,0.711674,0.594374,0.272845,0.890661,0.314224,1.0,0.961207,0.713843,0.635038,0.18908,0.913235,0.209483,1.0,0.963362,0.714022,0.641323,0.145366,0.931792,0.157112,1.0,0.967672,…,0.714255,0.645485,0.097773,0.940304,0.104741,1.0,0.971983,0.714404,0.64617,0.08399,0.941741,0.089778,1.0,0.971983,0.714404,0.646342,0.073707,0.944794,0.078556,1.0,0.974138,0.714463,0.646515,0.065613,0.945872,0.069828,1.0,0.974138,0.714463,0.646601,0.059397,0.951475,0.062845,1.0,0.976293,0.714506,0.646919
"""12_views""","""omnimatch_city_test""","""cl_optim""","""qwen3_06b-dj_adpt""",0.530172,0.183417,1.0,0.462151,0.530172,0.530172,0.183417,0.488793,0.829175,0.581897,0.96934,0.956897,0.70431,0.604632,0.291164,0.942543,0.314224,1.0,0.982759,0.707651,0.656789,0.202874,0.971217,0.209483,1.0,0.989224,0.708239,0.66485,0.155065,0.985494,0.157112,1.0,0.993534,…,0.708493,0.668149,0.103879,0.990368,0.104741,1.0,0.99569,0.708573,0.668532,0.089347,0.994947,0.089778,1.0,0.997845,0.708638,0.668852,0.078448,0.99921,0.078556,1.0,1.0,0.708698,0.669123,0.069732,0.99921,0.069828,1.0,1.0,0.708698,0.669123,0.062759,0.99921,0.062845,1.0,1.0,0.708698,0.669123
"""12_views""","""omnimatch_city_test""","""la2m_default""","""kalm_instruct_v2_5-dj_adpt""",0.540948,0.1862,1.0,0.462151,0.540948,0.540948,0.1862,0.462069,0.791532,0.581897,0.96934,0.93319,0.703843,0.58511,0.267888,0.88139,0.314224,1.0,0.956897,0.707192,0.624933,0.185201,0.905101,0.209483,1.0,0.961207,0.707541,0.631554,0.141918,0.921127,0.157112,1.0,0.965517,…,0.707788,0.635153,0.094899,0.922833,0.104741,1.0,0.965517,0.707788,0.635153,0.081466,0.925296,0.089778,1.0,0.967672,0.70785,0.635276,0.071659,0.930205,0.078556,1.0,0.969828,0.707908,0.635685,0.063985,0.932692,0.069828,1.0,0.969828,0.707908,0.635946,0.057931,0.936894,0.062845,1.0,0.971983,0.707951,0.636194
"""12_views""","""omnimatch_city_test""","""la2m_default""","""mini_l12-dj_adpt""",0.525862,0.179738,1.0,0.462151,0.525862,0.525862,0.179738,0.456034,0.761652,0.581897,0.96934,0.915948,0.686889,0.560692,0.267888,0.862788,0.314224,1.0,0.946121,0.691313,0.605908,0.185201,0.886826,0.209483,1.0,0.950431,0.691688,0.612564,0.139655,0.889921,0.157112,1.0,0.950431,…,0.691791,0.613549,0.093175,0.892076,0.104741,1.0,0.952586,0.691791,0.613549,0.079865,0.892076,0.089778,1.0,0.952586,0.691791,0.613549,0.069989,0.892923,0.078556,1.0,0.954741,0.691845,0.613578,0.062644,0.899876,0.069828,1.0,0.963362,0.692043,0.613853,0.056853,0.90529,0.062845,1.0,0.965517,0.692089,0.614191
"""12_views""","""omnimatch_city_test""","""cl_default""","""gtr_t5-dj_adpt""",0.540948,0.187125,1.0,0.462151,0.540948,0.540948,0.187125,0.480172,0.807637,0.581897,0.96934,0.943966,0.704095,0.592827,0.290302,0.930598,0.314224,1.0,0.978448,0.70879,0.649947,0.200144,0.956618,0.

In [4]:
for k in [10, 20]:
    for exp_group, datasets in [
        ("12_views", ["freyja", "omnimatch_city_test", "omnimatch_culture_test", "santos-union"]),
        ("08_views", ["webtable-union"]),
    ]:
        for dataset in datasets:
            per_view_means = per_view_means_dict[dataset]
            if len(per_view_means) == 0:
                continue

            exp_df = per_view_means.filter(pl.col("exp_group") == exp_group)

            # Max width: 241.14 pt
            fig, axes = plt.subplots(1, 2, figsize=(3.525, 1.5), layout="tight")
            for row_idx, metric in enumerate(["precision", "recall"]):
                ax: Axes = axes[row_idx]

                max_metric_value: float = exp_df[f"max_{metric}@{k}"].max()  # type: ignore[arg-type]
                ax.hlines(
                    y=max_metric_value,
                    xmin=-0.5,
                    xmax=3.5,
                    linestyles="--",
                    colors=COLOR_MAP["max_metric"],
                    alpha=0.8,
                )
                # ax.text(
                #     x=3.5,
                #     y=max_metric_value - 0.01,
                #     s=f"Max {metric}@{k}",
                #     ha="right",
                #     va="top",
                #     fontsize=5,
                #     color=colors["max_metric"],
                # )

                sns.boxplot(
                    data=exp_df,
                    x="method",
                    y=f"{metric}@{k}",
                    ax=ax,
                    order=["cl_optim", "la2m_default", "la2m_nopca", "v2v"],
                    hue="method",
                    hue_order=["cl_optim", "la2m_default", "la2m_nopca", "v2v"],
                    palette=COLOR_MAP,
                    linewidth=0.75,
                )
                ax.set_xlabel("")
                ax.set_ylabel(f"{metric.capitalize()}@{k}")
                ax.set_ylim(0, 1)
                ax.set_xticks(
                    [0, 1, 2, 3],
                    labels=[
                        r"\textsc{Fed}" + "\n" + r"\textsc{Augment}",
                        "LA2M",
                        "LA2M\n(no PCA)",
                        "vec2vec",
                    ],
                )

            _ = save_and_measure(fig, plots_dir / f"{exp_group}-{dataset}-k={k}.pdf")

analysis/plots/view-robustness/12_views-freyja-k=10.pdf: 239.760x93.959 pt (8.46x3.31 cm)
analysis/plots/view-robustness/12_views-omnimatch_city_test-k=10.pdf: 239.760x93.959 pt (8.46x3.31 cm)
analysis/plots/view-robustness/12_views-omnimatch_culture_test-k=10.pdf: 239.760x93.959 pt (8.46x3.31 cm)
analysis/plots/view-robustness/12_views-santos-union-k=10.pdf: 239.760x93.959 pt (8.46x3.31 cm)
analysis/plots/view-robustness/08_views-webtable-union-k=10.pdf: 239.760x93.959 pt (8.46x3.31 cm)
analysis/plots/view-robustness/12_views-freyja-k=20.pdf: 239.760x93.959 pt (8.46x3.31 cm)
analysis/plots/view-robustness/12_views-omnimatch_city_test-k=20.pdf: 239.760x93.959 pt (8.46x3.31 cm)
analysis/plots/view-robustness/12_views-omnimatch_culture_test-k=20.pdf: 239.760x93.959 pt (8.46x3.31 cm)
analysis/plots/view-robustness/12_views-santos-union-k=20.pdf: 239.760x93.959 pt (8.46x3.31 cm)
analysis/plots/view-robustness/08_views-webtable-union-k=20.pdf: 239.760x93.959 pt (8.46x3.31 cm)
